> **Portfolio summary:** Fine-tuning DistilBERT to classify credit-card complaints by policy category and resolution (code only; needs a GPU).
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

This notebook delves into the practical application of fine-tuning pre-trained language models for multi-label text classification, a critical task in areas like customer service and compliance. By leveraging DistilBERT, we aim to accurately categorize credit card complaints into their respective policy categories and resolutions. The approach demonstrates how a relatively small, task-specific dataset can be used to adapt powerful foundational models, enabling robust and automated analysis of unstructured text data. This exploration highlights the balance between model complexity and performance, offering insights into building efficient and effective NLP solutions.

In [ ]:
!pip install transformers datasets torch scikit-learn pandas numpy matplotlib seaborn -q
!pip install accelerate -q

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load dataset from HuggingFace Hub
dataset = load_dataset("priyaannamani/credit_card_qa", split="train")
df = dataset.to_pandas()

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample:")
df.head(3)

In [ ]:
# ============================================================
# SECTION 3: Explore & Preprocess Data

import pandas as pd

# Show key columns
print("=== Column Info ===")
print(df.dtypes)
print("\n=== Null Counts ===")
print(df.isnull().sum())

# Identify target columns (adjust if column names differ)
# Expected columns: 'complaint', 'policy_category', 'resolution'
# Print actual column names and samples to verify
for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(5))

In [ ]:
# ============================================================
# SECTION 4: Label Encoding & Sampling
# ============================================================
from sklearn.preprocessing import LabelEncoder
import numpy as np

# ---- Adjust column names if needed based on Section 3 output ----
COMPLAINT_COL   = 'complaint'        # input text column
POLICY_COL      = 'policy_category'  # target 1
RESOLUTION_COL  = 'resolution'       # target 2

# Drop rows with nulls in key columns
df_clean = df[[COMPLAINT_COL, POLICY_COL, RESOLUTION_COL]].dropna().reset_index(drop=True)
print(f"Cleaned dataset size: {len(df_clean)}")

# Label encode targets
le_policy = LabelEncoder()
le_resolution = LabelEncoder()

df_clean['policy_label']     = le_policy.fit_transform(df_clean[POLICY_COL])
df_clean['resolution_label'] = le_resolution.fit_transform(df_clean[RESOLUTION_COL])

print(f"\nPolicy categories ({len(le_policy.classes_)}): {le_policy.classes_}")
print(f"\nResolution classes ({len(le_resolution.classes_)}): {le_resolution.classes_}")

# ---- Sample 60 records for fine-tuning ----
SAMPLE_SIZE = 60
train_df = df_clean.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
eval_df  = df_clean.copy()  # evaluate on ALL records

print(f"\nTrain (fine-tune) size : {len(train_df)}")
print(f"Eval  (all records) size: {len(eval_df)}")

In [ ]:
# ============================================================
# SECTION 5: Build HuggingFace Dataset Objects

from datasets import Dataset

def make_hf_dataset(df, text_col, label_col):
    return Dataset.from_dict({
        'text':  df[text_col].tolist(),
        'label': df[label_col].tolist()
    })

# --- Task 1: Policy Category ---
train_policy = make_hf_dataset(train_df, COMPLAINT_COL, 'policy_label')
eval_policy  = make_hf_dataset(eval_df,  COMPLAINT_COL, 'policy_label')

# --- Task 2: Resolution ---
train_res = make_hf_dataset(train_df, COMPLAINT_COL, 'resolution_label')
eval_res  = make_hf_dataset(eval_df,  COMPLAINT_COL, 'resolution_label')

print("Train Policy dataset:", train_policy)
print("Eval  Policy dataset:", eval_policy)
print("Train Resolution dataset:", train_res)
print("Eval  Resolution dataset:", eval_res)

In [ ]:
# ============================================================
# SECTION 6: Fine-Tune Model 1 — Policy Category (DistilBERT)
# ============================================================
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME_BERT = "distilbert-base-uncased"
NUM_POLICY_LABELS = len(le_policy.classes_)

# --- Tokenizer ---
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)

def tokenize_bert(batch):
    return tokenizer_bert(
        batch['text'],
        truncation=True,
        max_length=256,
        padding=False
    )

train_policy_tok = train_policy.map(tokenize_bert, batched=True)
eval_policy_tok  = eval_policy.map(tokenize_bert, batched=True)

train_policy_tok.set_format('torch', columns=['input_ids','attention_mask','label'])
eval_policy_tok.set_format('torch',  columns=['input_ids','attention_mask','label'])

# --- Model ---
model_policy = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=NUM_POLICY_LABELS,
    ignore_mismatched_sizes=True
)

# --- Metrics ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted', zero_division=0)
    return {"accuracy": acc, "f1_weighted": f1}

# --- Training Arguments ---
training_args_policy = TrainingArguments(
    output_dir="./results_policy",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_dir="./logs_policy",
    logging_steps=10,
    warmup_steps=10,
    weight_decay=0.01,
    report_to="none",
    fp16=torch.cuda.is_available()
)

# --- Trainer ---
trainer_policy = Trainer(
    model=model_policy,
    args=training_args_policy,
    train_dataset=train_policy_tok,
    eval_dataset=eval_policy_tok,
    processing_class=tokenizer_bert,
    data_collator=DataCollatorWithPadding(tokenizer_bert),
    compute_metrics=compute_metrics
)

print("Starting Model 1 fine-tuning (Policy Category)...")
trainer_policy.train()
print("✅ Model 1 training complete!")

In [ ]:
# ============================================================
# SECTION 7: Fine-Tune Model 2 — Resolution (DistilBERT)
# ============================================================

NUM_RESOLUTION_LABELS = len(le_resolution.classes_)

# --- Tokenize ---
train_res_tok = train_res.map(tokenize_bert, batched=True)
eval_res_tok  = eval_res.map(tokenize_bert, batched=True)

train_res_tok.set_format('torch', columns=['input_ids','attention_mask','label'])
eval_res_tok.set_format('torch',  columns=['input_ids','attention_mask','label'])

# --- Model ---
model_resolution = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=NUM_RESOLUTION_LABELS,
    ignore_mismatched_sizes=True
)

# --- Training Arguments ---
training_args_res = TrainingArguments(
    output_dir="./results_resolution",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_dir="./logs_resolution",
    logging_steps=10,
    warmup_steps=10,
    weight_decay=0.01,
    report_to="none",
    fp16=torch.cuda.is_available()
)

# --- Trainer ---
trainer_resolution = Trainer(
    model=model_resolution,
    args=training_args_res,
    train_dataset=train_res_tok,
    eval_dataset=eval_res_tok,
    processing_class=tokenizer_bert,
    data_collator=DataCollatorWithPadding(tokenizer_bert),
    compute_metrics=compute_metrics
)

print("Starting Model 2 fine-tuning (Resolution)...")
trainer_resolution.train()
print("✅ Model 2 training complete!")

In [ ]:
# ============================================================
# SECTION 8: Baseline — Pre-trained DistilBERT (No Fine-Tuning)
# Zero-shot using random classification head
# ============================================================
from sklearn.dummy import DummyClassifier

# Majority class baseline (simple but fair comparison)
from sklearn.metrics import classification_report

y_policy_true = eval_df['policy_label'].values
y_res_true    = eval_df['resolution_label'].values

# --- Majority Class Baseline ---
dummy_policy = DummyClassifier(strategy='most_frequent')
dummy_policy.fit(train_df['policy_label'], train_df['policy_label'])
y_policy_dummy = dummy_policy.predict(y_policy_true)

dummy_res = DummyClassifier(strategy='most_frequent')
dummy_res.fit(train_df['resolution_label'], train_df['resolution_label'])
y_res_dummy = dummy_res.predict(y_res_true)

baseline_policy_acc = accuracy_score(y_policy_true, y_policy_dummy)
baseline_policy_f1  = f1_score(y_policy_true, y_policy_dummy, average='weighted', zero_division=0)

baseline_res_acc = accuracy_score(y_res_true, y_res_dummy)
baseline_res_f1  = f1_score(y_res_true, y_res_dummy, average='weighted', zero_division=0)

print("=== Majority Class Baseline ===")
print(f"Policy Category  — Accuracy: {baseline_policy_acc:.4f} | F1 (weighted): {baseline_policy_f1:.4f}")
print(f"Resolution       — Accuracy: {baseline_res_acc:.4f} | F1 (weighted): {baseline_res_f1:.4f}")

In [ ]:
# ============================================================
# SECTION 9: Evaluate Fine-Tuned Models on Full Dataset
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def get_predictions(trainer, dataset_tok):
    predictions = trainer.predict(dataset_tok)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids
    return preds, labels

# --- Model 1: Policy ---
print("Evaluating Model 1 (Policy Category)...")
preds_policy, true_policy = get_predictions(trainer_policy, eval_policy_tok)

acc_policy = accuracy_score(true_policy, preds_policy)
f1_policy  = f1_score(true_policy, preds_policy, average='weighted', zero_division=0)

print(f"\nModel 1 — Policy Category")
print(f"Accuracy     : {acc_policy:.4f}")
print(f"F1 (weighted): {f1_policy:.4f}")
print("\nClassification Report:")
print(classification_report(true_policy, preds_policy,
      target_names=le_policy.classes_, zero_division=0))

# --- Model 2: Resolution ---
print("\nEvaluating Model 2 (Resolution)...")
preds_res, true_res = get_predictions(trainer_resolution, eval_res_tok)

acc_res = accuracy_score(true_res, preds_res)
f1_res  = f1_score(true_res, preds_res, average='weighted', zero_division=0)

print(f"\nModel 2 — Resolution")
print(f"Accuracy     : {acc_res:.4f}")
print(f"F1 (weighted): {f1_res:.4f}")
print("\nClassification Report:")
print(classification_report(true_res, preds_res,
      target_names=le_resolution.classes_, zero_division=0))

In [ ]:
# ============================================================
# SECTION 10: Comparative Analysis — Charts
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# ---- 10A: Bar Chart — Accuracy & F1 Comparison ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fine-Tuned DistilBERT vs Majority Baseline", fontsize=14, fontweight='bold')

tasks = ['Policy Category', 'Resolution']
ft_acc  = [acc_policy, acc_res]
bl_acc  = [baseline_policy_acc, baseline_res_acc]
ft_f1   = [f1_policy, f1_res]
bl_f1   = [baseline_policy_f1, baseline_res_f1]

x = np.arange(len(tasks))
w = 0.35

# Accuracy
axes[0].bar(x - w/2, bl_acc, w, label='Baseline (Majority)', color='#b0b0b0', edgecolor='black')
axes[0].bar(x + w/2, ft_acc, w, label='Fine-Tuned DistilBERT', color='#3a7ebf', edgecolor='black')
axes[0].set_title('Accuracy Comparison')
axes[0].set_xticks(x); axes[0].set_xticklabels(tasks)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Score')
axes[0].legend()
for i, (b, f) in enumerate(zip(bl_acc, ft_acc)):
    axes[0].text(i - w/2, b + 0.02, f"{b:.3f}", ha='center', fontsize=9)
    axes[0].text(i + w/2, f + 0.02, f"{f:.3f}", ha='center', fontsize=9)

# F1
axes[1].bar(x - w/2, bl_f1, w, label='Baseline (Majority)', color='#b0b0b0', edgecolor='black')
axes[1].bar(x + w/2, ft_f1, w, label='Fine-Tuned DistilBERT', color='#e07b39', edgecolor='black')
axes[1].set_title('Weighted F1 Comparison')
axes[1].set_xticks(x); axes[1].set_xticklabels(tasks)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Score')
axes[1].legend()
for i, (b, f) in enumerate(zip(bl_f1, ft_f1)):
    axes[1].text(i - w/2, b + 0.02, f"{b:.3f}", ha='center', fontsize=9)
    axes[1].text(i + w/2, f + 0.02, f"{f:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("comparison_bar.png", dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")

# ---- 10B: Confusion Matrices ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Confusion Matrices — Fine-Tuned Models (Full Eval Set)", fontsize=13, fontweight='bold')

for ax, preds, trues, le, title in [
    (axes[0], preds_policy, true_policy, le_policy,     "Policy Category"),
    (axes[1], preds_res,    true_res,    le_resolution,  "Resolution")
]:
    cm = confusion_matrix(trues, preds)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=le.classes_,
                yticklabels=le.classes_,
                cmap='Blues', linewidths=0.5)
    ax.set_title(f"Model: {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()

# ---- 10C: Training Loss Curves ----
def plot_loss_curve(log_history, title, ax):
    train_loss = [(x['epoch'], x['loss']) for x in log_history if 'loss' in x and 'eval_loss' not in x]
    eval_loss  = [(x['epoch'], x['eval_loss']) for x in log_history if 'eval_loss' in x]
    if train_loss:
        ep, lo = zip(*train_loss)
        ax.plot(ep, lo, label='Train Loss', marker='o', color='steelblue')
    if eval_loss:
        ep, lo = zip(*eval_loss)
        ax.plot(ep, lo, label='Eval Loss', marker='s', linestyle='--', color='tomato')
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Training & Evaluation Loss Curves", fontsize=13, fontweight='bold')
plot_loss_curve(trainer_policy.state.log_history,     "Policy Category Model",  axes[0])
plot_loss_curve(trainer_resolution.state.log_history, "Resolution Model",       axes[1])
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SECTION 11: Summary Table
# ============================================================
import pandas as pd

summary = pd.DataFrame({
    'Task': ['Policy Category', 'Resolution'],
    'Model': ['DistilBERT (fine-tuned)', 'DistilBERT (fine-tuned)'],
    'Train Samples': [SAMPLE_SIZE, SAMPLE_SIZE],
    'Eval Samples': [len(eval_df), len(eval_df)],
    'Baseline Accuracy': [round(baseline_policy_acc, 4), round(baseline_res_acc, 4)],
    'FT Accuracy':       [round(acc_policy, 4), round(acc_res, 4)],
    'Baseline F1 (W)':   [round(baseline_policy_f1, 4), round(baseline_res_f1, 4)],
    'FT F1 (W)':         [round(f1_policy, 4), round(f1_res, 4)],
    'Accuracy Gain':     [round(acc_policy - baseline_policy_acc, 4),
                          round(acc_res   - baseline_res_acc, 4)],
})

print("=" * 80)
print("PERFORMANCE SUMMARY TABLE")
print("=" * 80)
print(summary.to_string(index=False))